# RNN/LSTM Time Series Forecasting — Armenia Poverty Rate

We use a 2-layer **LSTM** to forecast **`poverty_rate`** for each of Armenia's 11 regions (marzes) from the monthly panel data (2016–2022).

**Data:** `data/processed/panel/marz_monthly_panel_augmented.csv`  
- 924 rows: 11 regions × 84 months (Jan 2016 – Dec 2022)  
- Features: poverty_rate (lagged), crime_rate, hospitals, beds_per_10k, population  

**Split:**
- Train: 2016–2019 (48 months/region)
- Validation: 2020 (12 months/region, early stopping)
- Test: 2021–2022 (24 months/region)

**Baseline:** lag-1 model — predict poverty_rate(t) = poverty_rate(t−1)

In [ ]:
# Install PyTorch CPU (skip if already installed)
try:
    import torch
    print(f'PyTorch {torch.__version__} already installed')
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'torch', '--index-url', 'https://download.pytorch.org/whl/cpu',
                    '--quiet'], check=True)
    import torch
    print(f'PyTorch {torch.__version__} installed')

## 1. Load and prepare data

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path('..').resolve()
PANEL_CSV = PROJECT_ROOT / 'data' / 'processed' / 'panel' / 'marz_monthly_panel_augmented.csv'
OUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'results' / 'rnn'
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PANEL_CSV, low_memory=False, parse_dates=['date'])
df = df.sort_values(['marz', 'date']).reset_index(drop=True)

# Features used for forecasting
FEATURE_COLS = [
    'poverty_rate',
    'crime_rate_per_100k',
    'hospitals_per_100k',
    'beds_per_10k',
    'population',
]
TARGET = 'poverty_rate'

# Fill 1 NaN col (clinics) and drop any remaining NaN rows
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(method='ffill').fillna(method='bfill')

print(f'Panel shape: {df.shape}')
print(f'Regions ({df.marz.nunique()}): {sorted(df.marz.unique())}')
print(f'Year range: {df.year.min()} – {df.year.max()}')
print(f'Poverty rate: mean={df[TARGET].mean():.2f}%, std={df[TARGET].std():.2f}%')

## 2. Baseline — lag-1 predictor

In [ ]:
# Lag-1 baseline: predict poverty_rate(t) = poverty_rate(t-1) within each region
df['poverty_lag1'] = df.groupby('marz')['poverty_rate'].shift(1)
test_df = df[df['year'] >= 2021].dropna(subset=['poverty_lag1'])

r2_lag  = r2_score(test_df[TARGET], test_df['poverty_lag1'])
mae_lag = mean_absolute_error(test_df[TARGET], test_df['poverty_lag1'])
mse_lag = mean_squared_error(test_df[TARGET], test_df['poverty_lag1'])

print(f'Lag-1 Baseline (test 2021–2022)')
print(f'  R²  = {r2_lag:.4f}')
print(f'  MAE = {mae_lag:.4f}%')
print(f'  RMSE= {np.sqrt(mse_lag):.4f}%')

## 3. Sequence construction for LSTM

In [ ]:
SEQ_LEN = 12   # use 12 past months to predict next month

# Fit scaler on train data only
train_df  = df[df['year'] <= 2019]
val_df    = df[df['year'] == 2020]
test_df_f = df[df['year'] >= 2021]

scaler = StandardScaler()
scaler.fit(train_df[FEATURE_COLS].values)

def make_sequences(data_df, scaler, seq_len=SEQ_LEN):
    """Create (X, y) sliding-window sequences per region."""
    Xs, ys = [], []
    for region, grp in data_df.groupby('marz'):
        grp = grp.sort_values('date')
        arr = scaler.transform(grp[FEATURE_COLS].values)  # shape (T, F)
        tgt_idx = FEATURE_COLS.index(TARGET)
        for t in range(seq_len, len(arr)):
            Xs.append(arr[t - seq_len : t])           # (seq_len, F)
            ys.append(arr[t, tgt_idx])                # scalar (scaled target)
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

# For val/test we need the last seq_len months from the previous split
# Concatenate small context window before each split
def make_sequences_with_context(prev_df, curr_df, scaler, seq_len=SEQ_LEN):
    Xs, ys = [], []
    for region in curr_df['marz'].unique():
        prev_grp = prev_df[prev_df['marz'] == region].sort_values('date').tail(seq_len)
        curr_grp = curr_df[curr_df['marz'] == region].sort_values('date')
        combined = pd.concat([prev_grp, curr_grp]).sort_values('date')
        arr = scaler.transform(combined[FEATURE_COLS].values)
        tgt_idx = FEATURE_COLS.index(TARGET)
        offset = len(prev_grp)
        for t in range(seq_len, len(arr)):
            if t - seq_len >= offset or t - seq_len + seq_len > offset:
                Xs.append(arr[t - seq_len : t])
                ys.append(arr[t, tgt_idx])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_train, y_train = make_sequences(train_df, scaler)
X_val,   y_val   = make_sequences_with_context(train_df, val_df, scaler)
X_test,  y_test  = make_sequences_with_context(pd.concat([train_df, val_df]), test_df_f, scaler)

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}   | y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}  | y_test:  {y_test.shape}')

## 4. LSTM model definition

In [ ]:
class PovertyLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)        # (B, T, H)
        return self.fc(out[:, -1, :]).squeeze(-1)  # last timestep → scalar

INPUT_SIZE = len(FEATURE_COLS)
model = PovertyLSTM(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, dropout=0.2)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## 5. Training with early stopping

In [ ]:
BATCH_SIZE  = 32
MAX_EPOCHS  = 200
LR          = 1e-3
PATIENCE    = 20   # early stopping patience

# DataLoaders
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)),
    batch_size=BATCH_SIZE, shuffle=False
)

optimizer  = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
criterion  = nn.MSELoss()

train_losses, val_losses = [], []
best_val_loss = float('inf')
best_state    = None
patience_ctr  = 0

for epoch in range(1, MAX_EPOCHS + 1):
    # Train
    model.train()
    tl = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        tl += loss.item() * len(xb)
    tl /= len(train_loader.dataset)

    # Validate
    model.eval()
    vl = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            pred = model(xb)
            vl += criterion(pred, yb).item() * len(xb)
    vl /= len(val_loader.dataset)

    train_losses.append(tl); val_losses.append(vl)
    scheduler.step(vl)

    if vl < best_val_loss:
        best_val_loss = vl
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1

    if epoch % 20 == 0 or patience_ctr == 0:
        print(f'Epoch {epoch:4d} | train_loss={tl:.6f} | val_loss={vl:.6f}{" *" if patience_ctr == 0 else ""}')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

model.load_state_dict(best_state)
print(f'\nBest val loss: {best_val_loss:.6f}')

# Loss curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train loss', color='steelblue')
ax.plot(val_losses,   label='Val loss',   color='coral')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss (scaled)')
ax.set_title('LSTM Training and Validation Loss')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'lstm_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Test set evaluation and comparison vs baseline

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(torch.from_numpy(X_test)).numpy()

# Inverse-transform predictions and actuals (only the poverty_rate dimension)
tgt_idx = FEATURE_COLS.index(TARGET)
tgt_mean = scaler.mean_[tgt_idx]
tgt_std  = scaler.scale_[tgt_idx]

y_pred_pct = y_pred_scaled * tgt_std + tgt_mean
y_true_pct = y_test        * tgt_std + tgt_mean

r2_lstm   = r2_score(y_true_pct, y_pred_pct)
mae_lstm  = mean_absolute_error(y_true_pct, y_pred_pct)
rmse_lstm = np.sqrt(mean_squared_error(y_true_pct, y_pred_pct))

print('─' * 55)
print(f'{'Model':<20} {'R²':>8} {'MAE (%)':>12} {'RMSE (%)':>12}')
print('─' * 55)
print(f'{"Lag-1 Baseline":<20} {r2_lag:>8.4f} {mae_lag:>12.4f} {np.sqrt(mse_lag):>12.4f}')
print(f'{"LSTM (2-layer)":<20} {r2_lstm:>8.4f} {mae_lstm:>12.4f} {rmse_lstm:>12.4f}')
print('─' * 55)
print(f'LSTM vs Lag-1 — ΔR²: {r2_lstm - r2_lag:+.4f} | ΔMAE: {mae_lstm - mae_lag:+.4f}%')

# Save results
results = pd.DataFrame({
    'model':   ['Lag-1 Baseline', 'LSTM 2-layer'],
    'R2':      [r2_lag,   r2_lstm],
    'MAE_pct': [mae_lag,  mae_lstm],
    'RMSE_pct':[np.sqrt(mse_lag), rmse_lstm],
})
results.to_csv(OUT_DIR / 'lstm_vs_baseline.csv', index=False)
torch.save(best_state, OUT_DIR / 'lstm_best_weights.pt')

## 7. Visualization — predicted vs actual poverty rate (test period)

In [ ]:
# Scatter: predicted vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Poverty Rate Forecast vs Actual (Test: 2021–2022)', fontsize=13)

axes[0].scatter(y_true_pct, y_pred_pct, alpha=0.5, s=20, c='steelblue')
mn, mx = y_true_pct.min(), y_true_pct.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual Poverty Rate (%)')
axes[0].set_ylabel('Predicted Poverty Rate (%)')
axes[0].set_title(f'Predicted vs Actual (R²={r2_lstm:.4f})')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Residuals
residuals = y_true_pct - y_pred_pct
axes[1].hist(residuals, bins=40, color='coral', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', lw=1.5, ls='--')
axes[1].axvline(residuals.mean(), color='navy', lw=1.5, ls='-', label=f'Mean={residuals.mean():.2f}%')
axes[1].set_xlabel('Residual (Actual − Predicted, %)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'lstm_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Time-series plot: actual vs predicted for selected regions (2021-2022)
regions_to_plot = ['Yerevan', 'Shirak', 'Tavush', 'Syunik']
n = len(regions_to_plot)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('LSTM vs Actual Poverty Rate — Selected Regions (2021–2022)', fontsize=12)
axes = axes.ravel()

# Rebuild sequence-level metadata for test set
test_meta = []
for region in sorted(test_df_f['marz'].unique()):
    prev_grp = pd.concat([train_df, val_df])[pd.concat([train_df, val_df])['marz'] == region].sort_values('date').tail(SEQ_LEN)
    curr_grp = test_df_f[test_df_f['marz'] == region].sort_values('date')
    combined = pd.concat([prev_grp, curr_grp]).sort_values('date')
    for t in range(SEQ_LEN, len(combined)):
        row_date = combined.iloc[t]['date']
        test_meta.append({'marz': region, 'date': row_date})

meta_df = pd.DataFrame(test_meta)
meta_df['actual']    = y_true_pct
meta_df['predicted'] = y_pred_pct

for i, region in enumerate(regions_to_plot):
    if i >= n: break
    r_df = meta_df[meta_df['marz'] == region].sort_values('date')
    if len(r_df) == 0:
        axes[i].set_visible(False); continue
    axes[i].plot(r_df['date'], r_df['actual'],    label='Actual',    c='steelblue', lw=2)
    axes[i].plot(r_df['date'], r_df['predicted'], label='LSTM Pred', c='coral', lw=2, ls='--')
    axes[i].set_title(region)
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('Poverty Rate (%)')
    axes[i].legend(fontsize=8)
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'lstm_time_series_regions.png', dpi=150, bbox_inches='tight')
plt.show()